In [1]:
import glob
from general import *
from generation import *
from storage import *
from system import *
from copy import deepcopy

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
load_mw = Load_Data(years=2033).load_MW
#load_mw['load_MW'] = 20

In [4]:
load_mw

,load_MW
Date/time,
2033-01-01 00:30:00,61.873062
2033-01-01 01:30:00,62.962898
2033-01-01 02:30:00,62.851831
2033-01-01 03:30:00,62.807868
2033-01-01 04:30:00,64.001828
...,...
2033-12-31 19:30:00,63.699321
2033-12-31 20:30:00,63.543213
2033-12-31 21:30:00,63.384774


In [5]:
Kfiles = glob.glob('/projects/wg-ASGARD/weather_data/KAFB/*2013*')

In [6]:
KAFB_site = Site('KAFB',Weather_Data(Kfiles),-7)

In [7]:
ASGARD_PV = PV_System(name='ASGARD_PV',
                      site= KAFB_site,
                      capacity_MW_DC= 143,
                      PV_array_type= 'fixed',
                      PV_tilt=30,
                      PV_azimuth=180,
                      ratio_DC2AC=1.2,
                      off_grid_operation=True,
                      power_priority_load_MW_AC = None)

In [8]:
#modify CSP tower

In [9]:
csp_config = {'thermal_power_MW_t': 100,
          'power_rating_MW_e': 50,
          'dni_des':950,
          'tower_height_m': 117,
          'receiver_width_m':5,
          'receiver_height_m':5,
          'accept_ang_y':180,
          'accept_ang_x':180,
          'heliostat_width_m':5,
          'heliostat_height_m':5,
          'field_max_scaled_rad':8,
          'layout_method':'Radial Stagger',
          'field_shape':'Hexagon',
          'interaction_limit':50,
          'row_spacing_x':1.5,
          'row_spacing_y':1.5,
          'receiver_dT': 1,
          'media_cp': 1,
          'optical_height_m': 1,}

In [10]:
ASGARD_CSP = CSP_System(name='ASGARD_CSP',
                        site = KAFB_site,
                        config = csp_config,
                        off_grid_operation = True)

In [11]:
# loads the above csp system. will need to comment this block if you change the above.
# import pickle
# with open('/projects/wg-ASGARD/csp_obj/csp_100MWhth_50MW.pkl','rb') as file:
#     ASGARD_CSP = pickle.load(file)

In [12]:
# +

In [13]:
# # ASGARD_CSP.show_field()
# plt.figure(figsize=(10,5))

# plt.scatter(ASGARD_CSP.field['x_location'], ASGARD_CSP.field['y_location'], s=1.5)
# plt.xlabel('meters')
# plt.ylabel('meters')

In [14]:
ASGARD_TES = TES_System(name='ASGARD_TES',
                        site = KAFB_site,
                        capacity_MWh_e = 570,
                        power_rating_MW_e = 50,
                        power_minimum_MW_e = 0,
                        percent_discharge_depth = 95,
                        percent_heat_loss_daily = 1,
                        charge_rate_CSP_MW_t = .6*143,
                        charge_efficiency_t2TES = .98,
                        charge_rate_resistive_MW_e = None,
                        charge_efficiency_e2TES = .98,
                        systems_charging = ['ASGARD_CSP', 'ASGARD_PV2','ASGARD_PV', 'NSTTF_PV', 'Foxtail_PV'],
                        start_full = True,
                        off_grid_operation = True,
                        DOE_2030_targets = False,
                        dT_s = 200, # [C] dT across storage bins
                        cp_s = 1.15, # [kJ/kgK] storage media specific heat
                        rho_s = 2000) # [kg/m3] media bulk density)

In [15]:
ASGARD_BES = BES_System(name='ASGARD_BES',
                        site = KAFB_site,
                        capacity_MWh_e = 30,
                        power_rating_MW_e = 10,
                        percent_discharge_depth = 80,
                        charge_efficiency = .92,
                        discharge_efficiency = .92,
                        systems_charging = ['ASGARD_PV', 'NSTTF_PV', 'Foxtail_PV'],
                        start_full = True,
                        off_grid_operation = True,
                        max_total_power_MW = None)

In [16]:
csp = Power_System(
    name="power_csp",
    site = KAFB_site,
    power_type="thermal",
    energy_type="thermal",
    capacity_MW = 50,
    to_load = False,
    off_grid_operation= True,
    power_timeseries=[1] * 8760, #values actually set to pv values directly in system.py
    power_priority_load_MW = None,
    capex=150000000,  # USD
    opex=5000000,  # USD/year
    land_area=500  # Acres
)

In [17]:
bes = Storage_System(name='storage_bes',
                        site = KAFB_site,
                        capacity_MWh = 150,
                        power_type = 'thermal',
                        energy_type = 'thermal',
                        power_rating_MW = 35,
                        power_minimum_MW = 0,
                        baseload = True,
                        percent_discharge_depth = 80,
                        percent_loss_daily = 0,
                        charge_rate_MW = 1,
                        charge_efficiency = [0.92],
                        discharge_efficiency = .92,
                        conversion_values = [1] * 8760,
                        off_grid_operation = True,
                        start_full = True,
                        systems_charging = ['power_csp'])

In [18]:
ASGARD_systems = ['ASGARD_PV', 'ASGARD_TES','ASGARD_BES', 'ASGARD_CSP', 'bes', 'csp']
# ASGARD_systems = ['ASGARD_PV','ASGARD_CSP','ASGARD_TES','ASGARD_BES']

In [19]:
#need to change site for this configuration.
# ASGARD_CSP.site = KAFB_site

In [20]:
systems=ASGARD_systems
ASGARD = System(load_MW = load_mw,
             systems_load_order = [globals()[sys] for sys in systems])

8760
IRR: nan %
LCOE BT: 0.7392923929908064 $/kWh
Payback Period: None Years


In [21]:
ASGARD.system_metrics()

IRR: nan %
LCOE BT: 0.7392923929908064 $/kWh
Payback Period: None Years


{'PV_systems': ['ASGARD_PV'],
 'CSP_systems': ['ASGARD_CSP'],
 'BES_systems': ['ASGARD_BES'],
 'TES_systems': ['ASGARD_TES'],
 'Power_systems': ['power_csp'],
 'Storage_systems': ['storage_bes'],
 'off_grid_systems': ['ASGARD_PV',
  'ASGARD_CSP',
  'ASGARD_BES',
  'ASGARD_TES',
  'power_csp',
  'storage_bes'],
 'years': 1.0,
 'load_MWh': 629287.0581138763,
 'load_annual_MWh': [629287],
 'ASGARD_PV_capacity_MW_DC': 143,
 'PV_System_capacity_MW_DC': 143,
 'ASGARD_PV_capacity_MW_AC': 119.167,
 'PV_System_capacity_MW_AC': 262.16700000000003,
 'ASGARD_BES_capacity_MWh_e': 30,
 'BES_System_capacity_MWh_e': 30,
 'ASGARD_BES_capex_BES_capacity_USDpkWh': 184.37,
 'BES_System_capex_BES_capacity_USDpkWh': 214.37,
 'ASGARD_TES_capacity_MWh_e': 570,
 'TES_System_capacity_MWh_e': 570,
 'ASGARD_TES_capacity_MWh_t': 1017.0496267272607,
 'TES_System_capacity_MWh_t': 1587.0496267272606,
 'annual_electricity_sales_USD': [14698135],
 'ASGARD_TES_charge_rate_resistive_MW_e': 53.38268944239131,
 'ASGARD_TES

In [22]:
ts = ASGARD.timeseries

In [23]:
ts.columns

Index(['load_MW', 'target_load_MW', 'grid_to_load_MWh_e',
       'unmet_target_load_MWh_e', 'export_energy_MWh_e',
       'electricity_sale_in_hour', 'power_csp_power_MW',
       'power_csp_to_load_MWh', 'power_csp_to_grid_MWh',
       'power_csp_curtailed_MWh', 'KAFB_POI_MW',
       'power_csp_to_storage_bes_MWh', 'storage_bes_MWh',
       'storage_bes_to_load_MWh', 'storage_bes_loss_MWh',
       'ASGARD_PV_power_MW_AC', 'ASGARD_PV_to_load_MWh_e',
       'ASGARD_PV_to_grid_MWh_e', 'ASGARD_PV_curtailed_MWh_e',
       'ASGARD_PV_to_ASGARD_BES_MWh_e', 'ASGARD_PV_to_ASGARD_TES_MWh_e',
       'ASGARD_BES_MWh_DC', 'ASGARD_BES_to_load_MWh_e', 'ASGARD_CSP_heat_MW_t',
       'ASGARD_CSP_heat_unused_MWh_t', 'ASGARD_CSP_to_ASGARD_TES_MWh_t',
       'ASGARD_TES_MWh_t', 'ASGARD_TES_thermal_loss_MWh_t',
       'CSP_System_to_ASGARD_TES_MWh_t', 'ASGARD_TES_to_load_MWh_t',
       'ASGARD_TES_to_load_MWh_e', 'ASGARD_TES_to_load_MWh_gas_e',
       'ASGARD_TES_to_load_MWh_gas_t', 'ASGARD_TES_to_load_MWh

In [24]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
ts

,load_MW,target_load_MW,grid_to_load_MWh_e,unmet_target_load_MWh_e,export_energy_MWh_e,electricity_sale_in_hour,power_csp_power_MW,power_csp_to_load_MWh,power_csp_to_grid_MWh,power_csp_curtailed_MWh,KAFB_POI_MW,power_csp_to_storage_bes_MWh,storage_bes_MWh,storage_bes_to_load_MWh,storage_bes_loss_MWh,ASGARD_PV_power_MW_AC,ASGARD_PV_to_load_MWh_e,ASGARD_PV_to_grid_MWh_e,ASGARD_PV_curtailed_MWh_e,ASGARD_PV_to_ASGARD_BES_MWh_e,ASGARD_PV_to_ASGARD_TES_MWh_e,ASGARD_BES_MWh_DC,ASGARD_BES_to_load_MWh_e,ASGARD_CSP_heat_MW_t,ASGARD_CSP_heat_unused_MWh_t,ASGARD_CSP_to_ASGARD_TES_MWh_t,ASGARD_TES_MWh_t,ASGARD_TES_thermal_loss_MWh_t,CSP_System_to_ASGARD_TES_MWh_t,ASGARD_TES_to_load_MWh_t,ASGARD_TES_to_load_MWh_e,ASGARD_TES_to_load_MWh_gas_e,ASGARD_TES_to_load_MWh_gas_t,ASGARD_TES_to_load_MWh_gas_kg,ASGARD_TES_gas_dollars,ASGARD_TES_frac_nogas,ASGARD_TES_frac_gas,Power_System_to_storage_bes_MWh,unmet_load_MWh_e
Date/time,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2033-01-01 00:30:00,61.873062,None,0.000000,0.0,0,0.000000,0.0,0,0.0,0.0,0.000000,0.0,150.0,0.0,0,0.0,0.0,0,0,0.0,0.0,30.0,0.000000e+00,0.0,0.0,0,1017.049627,0.000000,0,0.000000,0.000000,0,0,0,0,0.000000,0.000000,NaN,NaN
2033-01-01 01:30:00,62.962898,None,0.000000,NaN,0,4407.402830,0.0,0,0.0,0.0,27.962898,0.0,115.0,35.0,0,0.0,0.0,0,0,0.0,0.0,30.0,0.000000e+00,0.0,0.0,0,966.021841,0.423771,0,50.604015,27.962898,0,0,0,0,0.559258,0.559258,0.0,0.000000
2033-01-01 02:30:00,62.851831,None,0.000000,NaN,0,4399.628205,0.0,0,0.0,0.0,27.851831,0.0,80.0,35.0,0,0.0,0.0,0,0,0.0,0.0,30.0,3.552714e-15,0.0,0.0,0,915.125628,0.402509,0,50.493704,27.851831,0,0,0,0,0.557037,0.557037,0.0,0.000000
2033-01-01 03:30:00,62.807868,None,0.000000,NaN,0,4396.550749,0.0,0,0.0,0.0,27.807868,0.0,45.0,35.0,0,0.0,0.0,0,0,0.0,0.0,30.0,0.000000e+00,0.0,0.0,0,864.205704,0.381302,0,50.538622,27.807868,0,0,0,0,0.556157,0.556157,0.0,0.000000
2033-01-01 04:30:00,64.001828,None,0.000000,NaN,0,4480.127974,0.0,0,0.0,0.0,49.001828,0.0,30.0,15.0,0,0.0,0.0,0,0,0.0,0.0,30.0,0.000000e+00,0.0,0.0,0,774.606434,0.360086,0,89.239184,49.001828,0,0,0,0,0.980037,0.980037,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2033-12-31 19:30:00,63.699321,None,63.699321,NaN,0,0.000000,0.0,0,0.0,0.0,0.000000,0.0,30.0,0.0,0,0.0,0.0,0,0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0,50.704347,0.021136,0,0.000000,0.000000,0,0,0,0,0.000000,0.000000,0.0,63.699321
2033-12-31 20:30:00,63.543213,None,63.543213,NaN,0,0.000000,0.0,0,0.0,0.0,0.000000,0.0,30.0,0.0,0,0.0,0.0,0,0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0,50.683220,0.021127,0,0.000000,0.000000,0,0,0,0,0.000000,0.000000,0.0,63.543213
2033-12-31 21:30:00,63.384774,None,63.384774,NaN,0,0.000000,0.0,0,0.0,0.0,0.000000,0.0,30.0,0.0,0,0.0,0.0,0,0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0,50.662102,0.021118,0,0.000000,0.000000,0,0,0,0,0.000000,0.000000,0.0,63.384774


In [25]:
ASGARD.metrics

{'PV_systems': ['ASGARD_PV'],
 'CSP_systems': ['ASGARD_CSP'],
 'BES_systems': ['ASGARD_BES'],
 'TES_systems': ['ASGARD_TES'],
 'Power_systems': ['power_csp'],
 'Storage_systems': ['storage_bes'],
 'off_grid_systems': ['ASGARD_PV',
  'ASGARD_CSP',
  'ASGARD_BES',
  'ASGARD_TES',
  'power_csp',
  'storage_bes'],
 'years': 1.0,
 'load_MWh': 629287.0581138763,
 'load_annual_MWh': [629287],
 'ASGARD_PV_capacity_MW_DC': 143,
 'PV_System_capacity_MW_DC': 143,
 'ASGARD_PV_capacity_MW_AC': 119.167,
 'PV_System_capacity_MW_AC': 262.16700000000003,
 'ASGARD_BES_capacity_MWh_e': 30,
 'BES_System_capacity_MWh_e': 30,
 'ASGARD_BES_capex_BES_capacity_USDpkWh': 184.37,
 'BES_System_capex_BES_capacity_USDpkWh': 214.37,
 'ASGARD_TES_capacity_MWh_e': 570,
 'TES_System_capacity_MWh_e': 570,
 'ASGARD_TES_capacity_MWh_t': 1017.0496267272607,
 'TES_System_capacity_MWh_t': 1587.0496267272606,
 'annual_electricity_sales_USD': [14698135],
 'ASGARD_TES_charge_rate_resistive_MW_e': 53.38268944239131,
 'ASGARD_TES